# Three clinical encounters: a scripted presenter rehearsal

**Educator description → structured JSON → student conversation → guided debrief.**

Choose **1** (pneumonia after Philadelphia hotel travel), **2** (tobacco use with no willingness to quit), or **3** (a 45-year-old woman with vague symptoms and an abnormal ECG).

This notebook uses the actual `/api/demo` HTTP handlers. Patient replies and debrief questions are **authored, static playback** representing the intended LLM-assisted experience. **No LLM is called, no latency is measured for a model, and no student is graded.** Entered questions advance sequence position, even if unrelated. Follow the presenter sequence for a coherent encounter.

The cases and individual values are synthetic and await clinical review. Urgent findings receive explicit escalation notices; keeping a debrief indirect must not withhold immediate safety guidance.

**Presenter:** “The repeatable script lets us rehearse the interface. Our existing live encounter pipeline is a separate mode.”

## 1. Choose the case and connection

Use your backend virtual environment as the notebook kernel. Install `backend_kelsch/requirements-notebooks.txt` there if needed. Local mode creates its own temporary SQLite database and does not use your cloud credentials. Your `.env` can stay in place.

For a running server, set `API_BASE_URL = "http://localhost:8000"`; enable `SCRIPTED_DEMO_ENABLED=true` on that server and apply its migration first. Both modes exercise the same routes.

In [ ]:
from pathlib import Path
import os, sys, json, time, tempfile, uuid, statistics
from html import escape
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets

CASE = 1                         # 1, 2, or 3
API_BASE_URL = None               # None = actual routes, isolated in-process SQLite
SHOW_NATURAL_DELAY = os.getenv("DEMO_NOTEBOOK_FAST") != "1"
REPLAY_ALL = True                 # Also show complete rehearsals for all three cases

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "backend/app/main.py").exists())
sys.path.insert(0, str(ROOT / "backend"))
import httpx
from app.settings import get_settings
from app.main import create_app
from app.db import Base, get_session
from app.models import tables
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from fastapi.testclient import TestClient

if API_BASE_URL:
    client = httpx.Client(base_url=API_BASE_URL, timeout=30)
    LOCAL = False
else:
    settings = get_settings()
    previous_demo_setting = settings.scripted_demo_enabled
    settings.scripted_demo_enabled = True
    db_path = Path(tempfile.mkdtemp(prefix="rounds-scripted-demo-")) / "rehearsal.sqlite"
    engine = create_engine(f"sqlite:///{db_path}", connect_args={"check_same_thread": False})
    Base.metadata.create_all(engine)
    app = create_app()
    def isolated_session():
        with Session(engine, expire_on_commit=False) as session:
            yield session
    app.dependency_overrides[get_session] = isolated_session
    client = TestClient(app)
    LOCAL = True

api_timings = []
def call(method, path, payload=None):
    started = time.perf_counter()
    response = client.request(method, path, **({"json": payload} if payload is not None else {}))
    api_timings.append({"path": path, "elapsed_ms": (time.perf_counter()-started)*1000,
                        "status": response.status_code})
    if response.is_error:
        raise RuntimeError(f"HTTP {response.status_code}: {response.json().get('message', 'Request failed')}")
    return response.json()

catalog = call("GET", "/api/demo/educator/scenarios")
selected = next(c for c in catalog if c["scenario_id"] == CASE)
display(Markdown(f"**Selected:** {selected['title']} · **Mode:** scripted playback · **Model calls:** 0"))

## 2. Educator mode: categorize a natural-language description

Change `EDUCATOR_TEXT`, or use the input below and click **Prepare description**. The structurer extracts explicit demographics and groups exact text spans into concerns, history, exposures, medicines, allergies, findings, and learning goals. It keeps negations and unclassified text.

**The extracted metadata and the fixed patient are separate JSON objects.** Differences are shown for review; pasted text never silently changes the static patient. To author a different patient, edit and version the fixture described at the end.

**Presenter:** “This JSON boundary can later accept a live model's structured output. Today the extraction is deterministic text rules.”

In [ ]:
EDUCATOR_TEXT = selected["example_description"]
prepared = None
sid = None

def prepare_description(text):
    global prepared, sid
    prepared = call("POST", "/api/demo/educator/preparations",
                    {"scenario_id": CASE, "description": text})
    sid = None  # Changing the educator input cannot silently reuse an old encounter.
    display(Markdown("### Fixed patient metadata"))
    display(HTML("<pre>" + escape(json.dumps(prepared["patient"], indent=2)) + "</pre>"))
    display(Markdown("### Extracted, source-linked metadata"))
    extraction = prepared["extracted_metadata"]
    display(pd.DataFrame(extraction["categorized_spans"]))
    for warning in prepared["warnings"]:
        display(Markdown("- " + warning))
    return prepared

prepare_description(EDUCATOR_TEXT)
editor = widgets.Textarea(value=EDUCATOR_TEXT, description="Educator:",
                         layout=widgets.Layout(width="95%", height="150px"))
prepare_button = widgets.Button(description="Prepare description")
prepare_output = widgets.Output()
def on_prepare(_):
    with prepare_output:
        clear_output(wait=True)
        try:
            prepare_description(editor.value)
        except Exception as exc:
            display(Markdown(str(exc)))
prepare_button.on_click(on_prepare)
display(editor, prepare_button, prepare_output)

## 3. Student mode: start the pinned encounter

Student responses contain the current transcript and only released chart items. They omit the educator's goal, future replies, full history, and instructor material. The educator and presenter can inspect the suggested sequence separately.

The patient is an actor in a fixed script: a question consumes the next reply. These endpoints do not interpret the question or replace the live production encounter API.

In [ ]:
def begin_selected():
    global sid
    if prepared is None:
        raise RuntimeError("Prepare the educator description first")
    state = call("POST", "/api/demo/sessions",
                 {"preparation_id": prepared["preparation_id"], "accept_fixed_script": True})
    sid = state["session_id"]
    return state

state = begin_selected()
display(pd.DataFrame([{"Patient": state["patient_name"], "Age": state["age_years"],
                       "Concern": state["presenting_concern"], "Scripted exchanges": state["total_steps"]}]))
display(Markdown("**Presenter-only suggested sequence**"))
display(pd.DataFrame({"Step": range(1, len(selected["suggested_student_sequence"])+1),
                      "Student question": selected["suggested_student_sequence"]}))

## 4. Enter a student question; receive the next static patient reply

Use the text box for this presenter notebook, or call `next_turn("your question")` in another cell. Click **Use suggested question**, then **Send question**, for the intended sequence. If educator text was changed, click **Start prepared case** first.

The response carries an assumed thinking delay of about 1–2.4 seconds. The notebook waits before displaying the patient bubble. The HTTP server does **not** sleep; the frontend controls presentation. Urgent chart information and notices appear immediately.

**Presenter:** “The wait is a presentation estimate, not measured Bedrock or voice latency.”

In [ ]:
def show_exchange(result, delay=True):
    t = result["turn"]
    if result["chart_added"]:
        display(pd.DataFrame(result["chart_added"])[["label", "text", "provenance"]])
    if result["safety_notice"]:
        display(Markdown("**Safety:** " + result["safety_notice"]))
    display(HTML('<div style="background:#e8edf5;padding:12px;border-radius:10px;margin:8px 0">'
                 '<b>Student clinician:</b> ' + escape(t["student_text"]) + '</div>'))
    if delay and SHOW_NATURAL_DELAY:
        time.sleep(t["simulated_delay_ms"] / 1000)
    display(HTML('<div style="background:#e3f4ed;padding:12px;border-radius:10px;margin:8px 0">'
                 '<b>Patient:</b> ' + escape(t["patient_text"]) + '</div>'))

def next_turn(text, request_id=None):
    global sid
    if sid is None:
        raise RuntimeError("Start the prepared case first")
    current = call("GET", f"/api/demo/sessions/{sid}")
    if current["next_step"] is None:
        raise RuntimeError("The sequence is complete or ended; use End and debrief")
    result = call("POST", f"/api/demo/sessions/{sid}/turns",
                  {"expected_step": current["next_step"],
                   "request_id": request_id or str(uuid.uuid4()), "text": text})
    show_exchange(result)
    return result

question = widgets.Textarea(description="Student:", layout=widgets.Layout(width="95%", height="85px"))
use_suggestion = widgets.Button(description="Use suggested question")
send_button = widgets.Button(description="Send question")
start_button = widgets.Button(description="Start prepared case")
end_button = widgets.Button(description="End and debrief")
conversation_output = widgets.Output()

def on_suggestion(_):
    with conversation_output:
        try:
            current = call("GET", f"/api/demo/sessions/{sid}") if sid else begin_selected()
            if current["next_step"]:
                question.value = selected["suggested_student_sequence"][current["next_step"]-1]
        except Exception as exc:
            display(Markdown(str(exc)))

def on_send(_):
    send_button.disabled = True
    with conversation_output:
        try:
            next_turn(question.value)
            question.value = ""
        except Exception as exc:
            display(Markdown(str(exc)))
    send_button.disabled = False

def show_debrief(session_id):
    call("POST", f"/api/demo/sessions/{session_id}/end")
    feedback = call("GET", f"/api/demo/sessions/{session_id}/debrief")
    display(Markdown("### Guided reflection — no grade"))
    display(Markdown(feedback["interpretation"]))
    for q in feedback["reflections"]:
        display(Markdown(f"**{q['domain_name']} · turn {q['anchor_step']}**"))
        display(HTML('<blockquote><b>Your words:</b> '+escape(q['student_quote'])+'</blockquote>'))
        display(Markdown(q["question"]))
    return feedback

def on_end(_):
    with conversation_output:
        try:
            show_debrief(sid)
        except Exception as exc:
            display(Markdown(str(exc)))

def on_start(_):
    with conversation_output:
        clear_output(wait=True)
        begin_selected()
        display(Markdown("Prepared case started. Use the suggested sequence."))

use_suggestion.on_click(on_suggestion)
send_button.on_click(on_send)
start_button.on_click(on_start)
end_button.on_click(on_end)
display(question, widgets.HBox([use_suggestion, send_button, start_button, end_button]), conversation_output)

## 5. Complete rehearsals for all three examples

Run this cell to see the whole conversation, chart releases, and debrief for each fixed case. These are new sessions, separate from the interactive encounter above.

**Clinical review questions:**

- Pneumonia: does the travel timeline support the investigation, while keeping the cause uncertain and treatment appropriately supervised?
- Tobacco: does the conversation respect refusal, and does the debrief explore the premature quit-date question without treating continued smoking as student failure?
- Cardiac: is testing and escalation early enough, and is prior anxiety handled without dismissing the current symptoms?

The ECG is a **synthetic textual teaching report**, not a generated waveform or validated ECG-reading exercise.

In [ ]:
rehearsals = []
if REPLAY_ALL:
    for scenario in catalog:
        display(Markdown(f"## Scenario {scenario['scenario_id']}: {scenario['title']}"))
        p = call("POST", "/api/demo/educator/preparations",
                 {"scenario_id": scenario["scenario_id"], "description": scenario["example_description"]})
        session = call("POST", "/api/demo/sessions",
                       {"preparation_id": p["preparation_id"], "accept_fixed_script": True})
        results = []
        for step, text in enumerate(scenario["suggested_student_sequence"], 1):
            result = call("POST", f"/api/demo/sessions/{session['session_id']}/turns",
                          {"expected_step": step, "request_id": f"rehearsal_{step}", "text": text})
            show_exchange(result)
            results.append(result)
        feedback = show_debrief(session["session_id"])
        rehearsals.append({"scenario": scenario, "session": session, "results": results, "debrief": feedback})
assert not REPLAY_ALL or len(rehearsals) == 3

## 6. Visualize the rehearsal, not learning effectiveness

The first panel counts fixed exchanges and reflection prompts. The second shows the **sum of assumed patient-display delays**. Neither is learner encounter duration, model performance, or evidence of educational benefit.

In [ ]:
summary = pd.DataFrame([
    {"Scenario": str(r["scenario"]["scenario_id"]),
     "Exchanges": len(r["results"]), "Reflection prompts": len(r["debrief"]["reflections"]),
     "Assumed display delay (s)": sum(t["turn"]["simulated_delay_ms"] for t in r["results"])/1000,
     "Actual model calls": sum(t["actual_model_calls"] for t in r["results"])}
    for r in rehearsals])
if len(summary):
    display(summary)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    summary.set_index("Scenario")[["Exchanges", "Reflection prompts"]].plot.bar(ax=axes[0], color=["#15375e", "#2a907b"])
    axes[0].set_title("Authored script structure")
    axes[0].set_ylabel("Count")
    axes[1].bar(summary["Scenario"], summary["Assumed display delay (s)"], color="#9770bd")
    axes[1].set_title("Assumed patient-display waits")
    axes[1].set_xlabel("Scenario")
    axes[1].set_ylabel("Seconds; not measured model latency")
    fig.tight_layout()
    plt.show()
display(Markdown(f"**Executed HTTP requests:** {len(api_timings)} · **Model calls:** 0 · **Scores:** none"))

## 7. Frontend connection contract

| Action | Method and endpoint | Body / response |
|---|---|---|
| Educator selects rehearsal | `GET /api/demo/educator/scenarios` | Goals, sample descriptions, suggested sequences; educator/presenter only |
| Structure educator text | `POST /api/demo/educator/preparations` | `{scenario_id, description}` → extracted spans, canonical patient, warnings, preparation ID |
| Start student mode | `POST /api/demo/sessions` | `{preparation_id, accept_fixed_script: true}` → public session |
| Resume | `GET /api/demo/sessions/{sid}` | Transcript, released chart, next step |
| Send committed student text | `POST /api/demo/sessions/{sid}/turns` | `{expected_step, request_id, text}` → fixed reply, new chart, safety notice, assumed delay |
| Finish | `POST /api/demo/sessions/{sid}/end` | Public ended session; early ending allowed |
| Reflect | `GET /api/demo/sessions/{sid}/debrief` | Transcript-linked reflection questions; no score |

**Frontend sequence:** preserve the session ID; disable send until the response returns; use one `request_id` per logical turn and reuse it for an identical retry. After a 409, reload the session. Display urgent chart/notices immediately, then optionally reveal the reply after its `simulated_delay_ms`. The server never delays urgent information.

Generated TypeScript contracts are in `frontend/src/api/schema.ts`; the typed `scriptedDemo` adapter is in `frontend/src/api/demo.ts`. Production spoken transcripts still use `/api/encounters/{id}/utterances`. A rehearsal text box belongs to a clearly labeled presenter surface; this PR adds no production student text box.

## 8. Replace the static ingredients later

- **Dialogue, findings, instructor goal, descriptions, references and reflection questions:** `backend/app/data/scripted_clinical_demo_v1.json`.
- **Interchangeable metadata contracts:** `backend/app/schemas/demo.py` (`PatientMetadata`, `MetadataExtraction`).
- **Categorization / sequence behavior:** `backend/app/demo/service.py`.
- **HTTP boundary:** `backend/app/api/demo.py`.

Increment `fixture_version` after an authored change. New preparations pin the new content and hash; existing sessions retain their saved snapshot. The fixture is a demo format, not the production `Case` schema. A live educator adapter can return `MetadataExtraction`, but must go through clinical authoring/review before changing real case facts. A RAG-guided debrief can later replace static prompts using approved hint material and explicit reveal rules; RAG is not used here.

References supporting the authored patterns: [CDC Legionella clinical guidance](https://www.cdc.gov/legionella/hcp/clinical-guidance/index.html), [CDC symptoms](https://www.cdc.gov/legionella/hcp/clinical-signs/index.html), [NCDHHS motivational interviewing](https://www.dph.ncdhhs.gov/programs/chronic-disease-and-injury/tobacco-prevention-and-control/quitlinenc/providers/treatment/motivational-interviewing), [2025 ACS guideline](https://www.jacc.org/doi/10.1016/j.jacc.2024.11.009), and [AHA symptoms in women](https://www.heart.org/en/health-topics/heart-attack/warning-signs-of-a-heart-attack/heart-attack-symptoms-in-women). These support the teaching context, not approval of these particular cases.

**Presenter closing:** “We can now rehearse the full interface consistently. Clinical approval, actual model/voice timing, and evidence that learners improve remain separate work.”